In [ ]:
# Install the Ultralytics YOLO library
!pip install ultralytics

# Check if NVIDIA GPU is available (This should show a table with 'Tesla T4' or similar)
!nvidia-smi

     ---------------------------------------- 1.1/1.1 MB 4.9 MB/s eta 0:00:00
     ---------------------------------------- 8.1/8.1 MB 6.2 MB/s eta 0:00:00
     ---------------------------------------- 3.7/3.7 MB 4.7 MB/s eta 0:00:00
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
  Using cached pillow-12.0.0-cp310-cp310-win_amd64.whl (7.0 MB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl (158 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
     -------------------------------------- 111.0/111.0 MB 4.3 MB/s eta 0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
     -------------------------------------- 802.4/802.4 kB 4.2 MB/s eta 0:00:00
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
     -------------------------------------- 121.8/121.8 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 1.6/1.6 MB 5.5 MB/s eta 0:00:00
  Using ca


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


All the images will be in the zip file named dataset.zip


In [ ]:
import os
from google.colab import files
import zipfile

# 1. Trigger the upload button
print("Please upload your dataset.zip file...")
uploaded = files.upload()

# 2. Unzip the dataset
zip_filename = list(uploaded.keys())[0] # Gets the name of the file you just uploaded
print(f"Unzipping {zip_filename}...")  

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset') # Extracts to a folder named 'dataset'

print("Dataset ready at /content/dataset")

In [ ]:
from ultralytics import YOLO

# 1. Load the model
# We use 'yolov8n.pt' (Nano) because it is the fastest for webcams.
model = YOLO('yolov8n.pt')

# 2. Start Training
# data: points to the config file inside your unzipped dataset
# epochs: 50 is usually enough for a small dataset. Change to 100 if accuracy is low.
# imgsz: 640 is standard resolution.
results = model.train(
    data='/content/dataset/data.yaml', 
    epochs=50, 
    imgsz=640,
    plots=True
)

print("Training Complete!")

after the training is done do the following 

In [ ]:
from google.colab import files

# The training results are saved in runs/detect/train/weights/
# 'best.pt' is the model with the highest accuracy.
files.download('/content/runs/detect/train/weights/best.pt')

In [ ]:
from IPython.display import Image, display
import os

# 1. Find the results folder (YOLO creates a new folder 'train', 'train2', etc. each time)
# We get the latest folder to ensure we see the most recent run.
runs_dir = '/content/runs/detect'
subfolders = [f.path for f in os.scandir(runs_dir) if f.is_dir()]
latest_run = max(subfolders, key=os.path.getmtime) # picks the newest folder

print(f"Displaying results from: {latest_run}")

# 2. Display the main Results Graph (Loss & Accuracy)
results_img = f"{latest_run}/results.png"
if os.path.exists(results_img):
    print("\n=== Training Loss & Accuracy Curves ===")
    display(Image(filename=results_img, width=800))
    print("What to look for:\n- 'box_loss' (Top Left): Should go DOWN.\n- 'mAP50' (Bottom Left): Should go UP (closer to 1.0 is better).")
else:
    print("Results image not found yet. Did training finish?")

# 3. Display the Confusion Matrix (Accuracy Breakdown)
conf_matrix_img = f"{latest_run}/confusion_matrix.png"
if os.path.exists(conf_matrix_img):
    print("\n=== Confusion Matrix ===")
    display(Image(filename=conf_matrix_img, width=600))
    print("What to look for:\n- A strong diagonal line from top-left to bottom-right is GOOD.\n- Any squares outside the diagonal mean the AI is confused.")